# Day 1 — Code Review Agent: End-to-End Test

This notebook validates every component of the Day 1 agent implementation:
- GroqClient LLM wrapper
- Individual tool smoke-tests
- Parser robustness tests
- Full end-to-end ReAct loop run
- Agent trace visualisation
- Performance summary

In [ ]:
# ── Cell 1: Imports, path setup, load .env ────────────────────────────────────
import sys
import os
import tempfile
from pathlib import Path

# Add project root to path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
print(f'Project root: {ROOT}')

from dotenv import load_dotenv
loaded = load_dotenv(ROOT / '.env')
print(f'Loaded .env: {loaded}')

from loguru import logger
logger.remove()
logger.add(sys.stdout, level='INFO', format='{time:HH:mm:ss} | {level} | {message}')

from configs.config import LLM, AGENT, TOOLS
print(f'Model: {LLM["model"]}')
print(f'Max steps: {AGENT["max_steps"]}')
print(f'API key set: {bool(LLM["api_key"])}')

In [ ]:
# ── Cell 2: Test GroqClient directly ─────────────────────────────────────────
from src.llm.groq_client import GroqClient

client = GroqClient()
print(f'GroqClient initialised. Model: {client.model}')

# Send a simple hello
response = client.complete(
    messages=[
        {'role': 'user', 'content': 'Say exactly: "Groq connection verified." and nothing else.'}
    ],
    max_tokens=20,
)
print(f'Response: {response}')

stats = client.get_usage_stats()
print(f'\nUsage stats:')
print(f'  Calls: {stats["total_calls"]}')
print(f'  Tokens used: {stats["total_tokens"]:,}')
print(f'  Estimated cost: ${stats["estimated_cost_usd"]:.4f} (free tier)')

assert 'verified' in response.lower() or len(response) > 0, 'No response received'
print('\n✓ GroqClient test passed')

In [ ]:
# ── Cell 3: Test each tool individually ──────────────────────────────────────
from src.tools.file_tools import read_file, list_python_files, get_function_context
from src.tools.analysis_tools import run_ruff, run_bandit, run_radon, check_imports

# Create a test file with known content
tmp = tempfile.NamedTemporaryFile(suffix='.py', delete=False, mode='w')
tmp.write('''
import os
import sys
from pathlib import *

PASSWORD = "hardcoded123"  # security issue

def complex_func(a, b, c, d, e):
    """Intentionally complex."""
    if a > 0:
        if b > 0:
            if c > 0:
                if d > 0:
                    return a + b + c + d
                else:
                    return a + b + c
            elif e > 0:
                return a + b + e
        else:
            return a
    return 0
''')
tmp.close()
test_path = tmp.name
print(f'Test file: {test_path}\n')

# --- read_file ---
print('=== read_file ===' )
result = read_file(test_path)
print(result[:300])
assert 'hardcoded123' in result
print('✓ read_file OK\n')

# --- read_file with line range ---
print('=== read_file (lines 1-3) ===')
result = read_file(test_path, start_line=1, end_line=3)
print(result)
print('✓ read_file range OK\n')

# --- run_ruff ---
print('=== run_ruff ===')
result = run_ruff(test_path)
print(result[:500])
print('✓ run_ruff OK\n')

# --- run_bandit ---
print('=== run_bandit ===')
result = run_bandit(test_path)
print(result[:500])
print('✓ run_bandit OK\n')

# --- run_radon ---
print('=== run_radon ===')
result = run_radon(test_path)
print(result[:500])
print('✓ run_radon OK\n')

# --- check_imports ---
print('=== check_imports ===')
result = check_imports(test_path)
print(result)
assert 'Wildcard' in result or 'wildcard' in result.lower(), 'Should detect wildcard import'
print('✓ check_imports OK\n')

# --- get_function_context ---
print('=== get_function_context ===')
result = get_function_context(test_path, 'complex_func')
print(result[:400])
assert 'complex_func' in result
print('✓ get_function_context OK')

import os as _os
_os.unlink(test_path)
print('\n✓ All tool tests passed')

In [ ]:
# ── Cell 4: Test the parser with malformed inputs ────────────────────────────
from unittest.mock import MagicMock
from src.agent.react_loop import ReActLoop
from src.agent.state import AgentState
from src.tools.registry import ToolRegistry

def make_loop():
    return ReActLoop(
        groq_client=MagicMock(),
        tool_registry=ToolRegistry(),
        prompt_engine=MagicMock(),
    )

def make_state():
    return AgentState(
        session_id='test',
        repo_url='https://github.com/test/repo',
        file_path='/tmp/test.py',
        file_content='x = 1',
    )

print('Testing _parse_response with various inputs...\n')

tests = [
    (
        'Valid well-formatted response',
        'Thought: I need to read the file.\nAction: read_file\nAction Input: {"file_path": "/tmp/x.py"}',
        True,
    ),
    (
        'Missing Action field',
        'Thought: Something.\nAction Input: {"file_path": "/tmp/x.py"}',
        False,
    ),
    (
        'Malformed JSON — unfixable',
        'Thought: t.\nAction: run_ruff\nAction Input: {file_path: /tmp, broken: }',
        False,
    ),
    (
        'Single-quoted JSON — fixable',
        "Thought: Check.\nAction: read_file\nAction Input: {'file_path': '/tmp/x.py'}",
        True,
    ),
    (
        'Trailing comma JSON — fixable',
        'Thought: Lint.\nAction: run_ruff\nAction Input: {"file_path": "/tmp/x.py",}',
        True,
    ),
    (
        'Bare .py path as Action Input',
        'Thought: Read.\nAction: read_file\nAction Input: /tmp/example.py',
        True,
    ),
    (
        'finish_review action',
        'Thought: Done.\nAction: finish_review\nAction Input: {"summary": "Found 3 issues.", "detailed_review": "## Review"}',
        True,
    ),
    (
        'Completely garbled response',
        'Lorem ipsum dolor sit amet consectetur adipiscing elit',
        False,
    ),
]

all_passed = True
for description, response, expect_success in tests:
    loop = make_loop()
    state = make_state()
    result = loop._parse_response(response, state)
    success = result is not None
    status = '✓' if success == expect_success else '✗'
    if success != expect_success:
        all_passed = False
    expected_str = 'success' if expect_success else 'None'
    got_str = 'success' if success else 'None'
    print(f'{status} [{description}]')
    print(f'  expected={expected_str}, got={got_str}')
    if result and success:
        thought, action, action_input = result
        print(f'  → action={action}, input={action_input}')
    print()

print('✓ All parser tests passed' if all_passed else '✗ Some parser tests FAILED')

In [ ]:
# ── Cell 5: Full end-to-end agent run ────────────────────────────────────────
from src.agent.prompt_engine import PromptEngine
from src.agent.react_loop import ReActLoop
from src.llm.groq_client import GroqClient
from src.tools.registry import ToolRegistry
from src.tools.file_tools import register_file_tools
from src.tools.analysis_tools import register_analysis_tools
from src.tools.defect_api_tool import register_defect_api_tools

# Create a test file with security issues
TEST_CODE = '''
import os
import requests

SECRET_KEY = "hardcoded-secret-abc123"  # CWE-798
DB_PASS    = "password123"              # CWE-798

def login(conn, username, password):
    """Authenticate user — SQL injection risk."""
    query = "SELECT id FROM users WHERE username='" + username + "' AND password='" + password + "'"
    cur = conn.cursor()
    cur.execute(query)
    return cur.fetchone()

def run_command(user_input):
    """Execute user-supplied command — command injection."""
    return eval(user_input)   # B307

def fetch_data(url):
    """HTTP request with SSL verification disabled."""
    return requests.get(url, verify=False)  # B501
'''

import tempfile, os
f = tempfile.NamedTemporaryFile(suffix='.py', delete=False, mode='w')
f.write(TEST_CODE)
f.close()
test_file = f.name
print(f'Test file: {test_file}\n')

# Build agent
groq_client   = GroqClient()
registry      = ToolRegistry()
prompt_engine = PromptEngine()

register_file_tools(registry)
register_analysis_tools(registry)
register_defect_api_tools(registry)

loop = ReActLoop(groq_client, registry, prompt_engine)

print('Running agent...')
state = loop.run(
    file_path=test_file,
    repo_url='https://github.com/example/vulnerable-app',
    file_content=TEST_CODE,
    risk_score=0.92,
    risk_label='HIGH',
    shap_features=[
        {'feature_name': 'hardcoded_secrets', 'feature_value': 2,  'shap_value': 0.45},
        {'feature_name': 'eval_calls',        'feature_value': 1,  'shap_value': 0.38},
        {'feature_name': 'sql_concat',        'feature_value': 1,  'shap_value': 0.31},
    ],
)

print(f'\n✓ Agent completed in {state.elapsed_seconds:.1f}s')
print(f'  Status:  {state.status.value}')
print(f'  Steps:   {state.current_step}')
print(f'  Issues:  {len(state.issues_found)}')

print('\n' + '='*60)
print('COMPLETE REACT TRACE')
print('='*60)
for step in state.thought_history:
    print(f'\n[Step {step.step_number}]')
    print(f'  Thought: {step.thought[:200]}')
    print(f'  Action:  {step.action}')
    print(f'  Input:   {step.action_input}')
    obs_short = step.observation[:300].replace('\n', ' ')
    print(f'  Obs:     {obs_short}...' if len(step.observation) > 300 else f'  Obs:     {obs_short}')

print('\n' + '='*60)
print('FINAL REVIEW')
print('='*60)
print(state.final_review or '(no final review produced)')

os.unlink(test_file)

In [ ]:
# ── Cell 6: Visualise agent trace as timeline diagram ────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from pathlib import Path

if not state.thought_history:
    print('No trace to visualise — run Cell 5 first.')
else:
    # Colour by tool category
    CATEGORY_COLORS = {
        'file':       '#4C72B0',   # blue
        'analysis':   '#DD8452',   # orange  
        'control':    '#55A868',   # green
        'defect_api': '#C44E52',   # red
        'memory':     '#8172B2',   # purple
        'unknown':    '#666666',   # grey
    }

    tool_category = {}
    for name, tool in registry._tools.items():
        tool_category[name] = tool.category

    steps = [s.step_number for s in state.thought_history]
    tools = [s.action      for s in state.thought_history]
    unique_tools = sorted(set(tools))
    tool_y = {t: i for i, t in enumerate(unique_tools)}

    ys     = [tool_y[t] for t in tools]
    colors = [CATEGORY_COLORS.get(tool_category.get(t, 'unknown'), '#666666') for t in tools]
    thoughts_short = [
        s.thought[:50] + '...' if len(s.thought) > 50 else s.thought
        for s in state.thought_history
    ]

    fig, ax = plt.subplots(figsize=(14, max(5, len(unique_tools) * 1.1 + 2)))

    # Plot steps
    for x, y, c, thought in zip(steps, ys, colors, thoughts_short):
        ax.scatter(x, y, s=200, color=c, zorder=5, edgecolors='white', linewidths=1.5)
        ax.annotate(
            thought,
            xy=(x, y), xytext=(0, 14),
            textcoords='offset points',
            fontsize=7, ha='center', va='bottom',
            color='#333333',
            rotation=25,
        )

    # Connect steps with light line
    ax.plot(steps, ys, color='#cccccc', linewidth=1, zorder=1)

    # Axis formatting
    ax.set_yticks(list(tool_y.values()))
    ax.set_yticklabels(list(tool_y.keys()), fontsize=9)
    ax.set_xlabel('Step Number', fontsize=11)
    ax.set_title(
        f'Agent Trace — {Path(state.file_path).name}\n'
        f'{state.current_step} steps · {state.elapsed_seconds:.1f}s · '
        f'status={state.status.value}',
        fontsize=12,
    )
    ax.set_xlim(-0.5, max(steps) + 1)
    ax.grid(axis='x', linestyle='--', alpha=0.4)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # Legend
    patches = [
        mpatches.Patch(color=v, label=k)
        for k, v in CATEGORY_COLORS.items()
        if k in set(tool_category.get(t, 'unknown') for t in tools)
    ]
    ax.legend(handles=patches, title='Tool Category', loc='lower right', fontsize=8)

    plt.tight_layout()
    out_dir = Path(ROOT) / 'plots'
    out_dir.mkdir(exist_ok=True)
    out_path = out_dir / 'agent_trace.png'
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'\nSaved to: {out_path}')

In [ ]:
# ── Cell 7: Day 1 Summary ─────────────────────────────────────────────────────
from src.llm.groq_client import GroqClient

usage  = groq_client.get_usage_stats()
counts = state.issue_count_by_severity

print('=' * 60)
print('DAY 1 SUMMARY')
print('=' * 60)
print(f'  Status:          {state.status.value}')
print(f'  Steps taken:     {state.current_step}')
print(f'  Tools called:    {", ".join(sorted(set(state.tools_called)))}')
print(f'  Issues found:    {len(state.issues_found)}')
for sev, cnt in counts.items():
    if cnt:
        print(f'    {sev}: {cnt}')
print(f'  LLM calls:       {usage["total_calls"]}')
print(f'  Total tokens:    {usage["total_tokens"]:,}')
print(f'  Time elapsed:    {state.elapsed_seconds:.1f}s')
print(f'  Cost:            ${usage["estimated_cost_usd"]:.4f} (free tier)')
print()
print('Git commit:')
print('  git add .')
print('  git commit -m "feat: Day 1 — ReAct agent core complete"')